# 1.04 — Integración del target: salidas por estación y hora

**Proyecto:** MiBici — TC5044.10 — Equipo 04
**Autor:** Leonardo Santos García
**Fecha:** 24 de octubre de 2026
**Rama:** `feat/pr2-integration`

> **Problema de negocio:** pronosticar viajes iniciados por estación y por hora para el día siguiente, para apoyar la planeación del rebalanceo de bicicletas.

Este notebook **no construye datos**: lee las salidas del pipeline (`make interim`, `make station-hour`), las valida visualmente, integra el EDA del equipo y documenta las decisiones que dan forma a `processed_v1`. La lógica vive en `mibici/` y está probada en `tests/`.

## 1. Objetivo

1. Verificar que la base `station_hour_base` representa correctamente el target: una fila por estación y hora **activa**, con ceros solo donde la estación existía.
2. Integrar los hallazgos de los notebooks de EDA del equipo (1.01, 1.02, 1.03).
3. Tomar y justificar las decisiones abiertas (periodo, flags, estaciones nuevas, días atípicos).
4. Documentar `processed_v1` y su registro en MLflow.

## 2. Datos y `station_hour_base`

| Archivo | Generado por | Contenido |
|---|---|---|
| `data/interim/station_hour_base.parquet` | `make station-hour` | Salidas por estación × hora activa, con columnas de calendario |
| `data/interim/stations.parquet` | `make interim` | Ciclo de vida por estación (primer/último mes, meses inactivos, nueva, retirada) |
| `reports/station_hour/summary.json` | `make station-hour` | Conteos de validación de la base |

**Definición de salida:** un viaje contado en `origin_station_id` a la hora de `started_at`. Se usan todos los viajes de `interim_v1` (los flags no se excluyen en esta base).

**Ventana activa de una estación:** del día de su primer viaje al día de su último viaje (como origen o destino), sin los meses en que no tuvo ningún viaje.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from mibici.config import FIGURES_DIR, INTERIM_DATA_DIR, REPORTS_DIR

# Estilo de las figuras: marcas delgadas, ejes y rejilla discretos
SERIES = "#2a78d6"  # color de la serie (una sola serie por figura)
MUTED = "#52514e"  # texto secundario y ejes
INACTIVE = "#d9d8d4"  # bandas de periodos inactivos
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": MUTED,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": "#e6e5e1",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,  # rejilla detrás de las barras
    }
)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

base = pd.read_parquet(INTERIM_DATA_DIR / "station_hour_base.parquet")
stations = pd.read_parquet(INTERIM_DATA_DIR / "stations.parquet").set_index("station_id")
summary = json.loads((REPORTS_DIR / "station_hour" / "summary.json").read_text())

print(f"Filas: {len(base):,} | estaciones: {base['station_id'].nunique()}")
base.dtypes.to_frame("dtype")

In [ ]:
base.head()

In [ ]:
pd.Series(summary, name="valor").to_frame()

**Interpretación**

- **28,246,536 filas** = suma, sobre las 486 estaciones, de 24 × días activos.
- **`departures_total` = `n_trips_interim` = 39,078,226**: toda salida de interim cayó en una hora de la grilla; no se perdió ningún viaje al construirla.
- **`duplicated_keys` = 0**: cada (estación, hora) aparece una sola vez.
- **54.4 % de ceros válidos**: en más de la mitad de las horas activas no sale ninguna bicicleta. El target es un **conteo con muchos ceros** (mediana 0, p90 = 4, p99 = 11), lo que importa para elegir la métrica y el modelo en fases posteriores (por ejemplo, el error porcentual no está definido cuando el valor real es 0).

### 2.1 Validación formal: contrato de la base

Revisar los números a ojo no basta. La siguiente celda expresa el **contrato** que debe cumplir `station_hour_base` como afirmaciones (`assert`): si alguna falla, el notebook se detiene con un mensaje claro. Es la misma idea que una compuerta de calidad en un pipeline antes de entrenar.

In [ ]:
EXPECTED_COLUMNS = [
    "station_id", "hour_start", "departures", "date", "hour", "day_of_week", "is_weekend", "month",
]

# 1) Esquema: columnas esperadas, en orden
assert list(base.columns) == EXPECTED_COLUMNS, f"Columnas inesperadas: {list(base.columns)}"
# 2) Sin faltantes en ninguna columna
assert base.notna().all().all(), "Hay valores faltantes"
# 3) Target: conteo entero y no negativo
assert pd.api.types.is_integer_dtype(base["departures"]), "departures debe ser entero"
assert (base["departures"] >= 0).all(), "Hay salidas negativas"
# 4) Clave única: una fila por estación y hora
assert not base.duplicated(["station_id", "hour_start"]).any(), "Claves duplicadas"
# 5) Conservación: toda salida de interim está en la grilla
assert base["departures"].sum() == summary["n_trips_interim"], "Se perdieron o duplicaron salidas"
# 6) Calendario coherente con hour_start
assert base["hour"].between(0, 23).all() and base["day_of_week"].between(0, 6).all()
assert (base["is_weekend"] == base["day_of_week"].isin([5, 6])).all()
# 7) Toda estación de la base tiene ciclo de vida en stations.parquet
assert base["station_id"].isin(stations.index).all(), "Estaciones sin ciclo de vida"

print("Contrato de station_hour_base: 7 verificaciones OK")

## 3. Validación de ceros: la grilla respeta el ciclo de vida

Regla:

| Situación | ¿Fila en la base? |
|---|---|
| Estación activa sin salidas en esa hora | Sí, con `departures = 0` (cero válido) |
| Antes del primer viaje de la estación | No |
| Mes completo sin viajes dentro de su vida (inactiva) | No |
| Después de su último viaje (retirada) | No |

Se revisa una estación de cada tipo: consolidada, nueva de 2026, con meses inactivos y retirada.

In [ ]:
# Una estación de cada tipo (elegidas en la validación del bloque 2)
examples = {2: "consolidada", 436: "nueva 2026", 94: "con meses inactivos", 22: "retirada (ex-)"}

rows = []
for station_id, kind in examples.items():
    grid = base[base["station_id"] == station_id]
    info = stations.loc[station_id]
    rows.append(
        {
            "tipo": kind,
            "station_id": station_id,
            "primera_hora_grilla": grid["hour_start"].min(),
            "ultima_hora_grilla": grid["hour_start"].max(),
            "meses_en_grilla": grid["hour_start"].dt.to_period("M").nunique(),
            "meses_activos": info["n_active_months"],
            "meses_inactivos": info["n_inactive_months"],
            "salidas_grilla": int(grid["departures"].sum()),
            "viajes_origen_interim": info["n_trips_origin"],
            "pct_ceros": round((grid["departures"] == 0).mean() * 100, 1),
        }
    )
pd.DataFrame(rows).set_index("tipo")

**Interpretación**

- En las cuatro estaciones, **las salidas de la grilla coinciden exactamente con sus viajes de origen en interim**.
- **Nueva (436):** la grilla empieza el **3 de junio de 2026**, el día de su primer viaje, no el 1 de junio. Es el efecto de usar bordes por día: 285 estaciones abrieron después del día 10 de su primer mes y, con bordes por mes, habrían recibido ceros falsos.
- **Con meses inactivos (94):** sus 137 meses de vida se reducen a **80 meses en la grilla**; los **57 meses inactivos** no generan filas. Rellenarlos con 0 habría reducido su promedio de salidas aproximadamente a la mitad.
- **Retirada (22):** la grilla termina el día de su último viaje (2018-05-30); no hay filas posteriores.
- **Principio aplicado — nunca imputar una etiqueta:** `departures` es el target. Rellenar con 0 un periodo en que la estación no existía equivale a **fabricar etiquetas**: ejemplos de entrenamiento que nunca ocurrieron. Por eso esas horas no se crean, en lugar de imputarse.
- El porcentaje de ceros varía mucho entre estaciones (de ~44 % a ~71 %): el volumen por estación es heterogéneo.

In [ ]:
# Estación 94: salidas por mes en toda su vida; los meses inactivos se marcan como bandas
station_id = 94
grid = base[base["station_id"] == station_id]
monthly = grid.groupby(grid["hour_start"].dt.to_period("M"))["departures"].sum()

# Todos los meses entre el primero y el último; los que no están en la grilla son inactivos
all_months = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
monthly = monthly.reindex(all_months)
inactive = monthly.isna()

fig, ax = plt.subplots(figsize=(10, 3.6))
x = all_months.to_timestamp()
for month in x[inactive.to_numpy()]:
    ax.axvspan(month, month + pd.offsets.MonthBegin(1), color=INACTIVE, linewidth=0)
ax.bar(x, monthly.fillna(0), width=25, color=SERIES, align="edge")
ax.set_title(
    f"Estación {station_id}: salidas por mes (bandas grises = meses inactivos, sin filas en la base)",
    loc="left",
    fontsize=11,
)
ax.set_ylabel("Salidas por mes")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "1.04-lsg-station-94-lifecycle.png", dpi=150)
plt.show()

**Interpretación**

La estación 94 alterna periodos de operación con periodos largos sin ningún viaje (bandas grises). Esos meses **no aparecen en la base**: no son demanda cero, son periodos en que la estación no estaba operando. Si se rellenaran con 0, el modelo aprendería que esta estación casi no tiene demanda y recomendaría enviar menos bicicletas justo cuando sí opera.

In [ ]:
# Distribución del target: proporción de horas activas por número de salidas
CAP = 15  # 15 o más se agrupan en la última barra
values = base["departures"].clip(upper=CAP)
share = values.value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(share.index, share.values, width=0.8, color=SERIES)
ax.set_xticks(range(CAP + 1), [str(v) for v in range(CAP)] + [f"{CAP}+"])
ax.set_xlabel("Salidas en la hora")
ax.set_ylabel("% de horas activas")
ax.set_title("Distribución de salidas por estación y hora (todas las horas activas)", loc="left", fontsize=11)
ax.annotate(
    f"{share.loc[0]:.1f} % de ceros válidos",
    xy=(0, share.loc[0]),
    xytext=(1.2, share.loc[0] * 0.9),
    color=MUTED,
    fontsize=10,
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "1.04-lsg-departures-distribution.png", dpi=150)
plt.show()

share.round(2).to_frame("% de horas").T

**Interpretación**

El target está **muy concentrado en valores bajos**: más de la mitad de las horas activas tienen 0 salidas y la frecuencia cae rápidamente al aumentar el conteo, con una cola larga (máximo 207). Implicaciones para fases posteriores:

- Es un problema de **conteos con exceso de ceros**; un baseline razonable debe compararse contra predecir siempre 0 o el promedio histórico de la misma estación y hora.
- Las métricas porcentuales (MAPE) no sirven cuando el valor real es 0; conviene MAE/RMSE o métricas para conteos.
- La cola larga sugiere revisar eventos especiales (celda de horas máximas, más abajo).

### 3.1 Estadísticas del target y sobredispersión

**¿Qué se quiere saber?** Si la variabilidad del target es la que esperaría un modelo de conteos simple, o mucho mayor.

**Fundamento.** El modelo de conteos más simple es la distribución de **Poisson**, que asume que la varianza es igual a la media:

$$\text{Poisson:}\quad \operatorname{Var}(Y) = \mathbb{E}[Y] \qquad\Longrightarrow\qquad D = \frac{\operatorname{Var}(Y)}{\mathbb{E}[Y]} = 1$$

- $Y$: salidas de una estación en una hora.
- $\mathbb{E}[Y]$: promedio de salidas; $\operatorname{Var}(Y)$: varianza.
- $D$: **índice de dispersión**. Se lee: *"cuántas veces más varía el conteo de lo que asumiría Poisson"*.

$D \approx 1$ → Poisson es razonable. $D > 1$ → **sobredispersión**: hay más horas en 0 y más picos de lo que Poisson permite (por ejemplo, por diferencias entre estaciones, horas pico o eventos).

**Ejemplo sencillo:** cuatro horas con salidas 0, 0, 0 y 8 tienen media 2 y varianza (muestral) 16 → $D = 8$: mucho más irregular que cuatro horas con 2, 2, 2 y 2 ($D = 0$).

In [ ]:
departures = base["departures"]
target_stats = pd.Series(
    {
        "media": departures.mean(),
        "varianza": departures.var(),
        "indice_dispersion (var/media)": departures.var() / departures.mean(),
        "asimetria (skewness)": departures.skew(),
        "pct_ceros": (departures == 0).mean() * 100,
    }
).round(3)
target_stats.to_frame("todas las horas activas")

In [ ]:
# El mismo análisis por estación: ¿la sobredispersión es solo por mezclar estaciones distintas?
per_station = base.groupby("station_id")["departures"].agg(
    media="mean", varianza="var", pct_ceros=lambda x: (x == 0).mean() * 100
)
per_station["indice_dispersion"] = per_station["varianza"] / per_station["media"]
per_station[["media", "pct_ceros", "indice_dispersion"]].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
).round(2)

**Interpretación**

- Con todas las horas activas: media ≈ 1.38, varianza ≈ 6.36 → **índice de dispersión ≈ 4.6** y asimetría ≈ 4.3 (cola larga a la derecha).
- **Por estación**, el índice baja pero **todas** las estaciones siguen por encima de 1 (mínimo ≈ 1.12, mediana ≈ 2.0). Parte de la sobredispersión global se explica por **mezclar estaciones con volúmenes muy distintos** (promedio por hora desde ≈ 0.01 hasta ≈ 8.1), pero dentro de cada estación sigue habiendo más variación que la de Poisson: la hora del día, el día de la semana y los eventos importan.
- El porcentaje de ceros también varía mucho entre estaciones (mediana ≈ 60 %, de ≈ 25 % a ≈ 100 %).

**Implicaciones para la Fase 2 (no se deciden aquí):**

1. El modelo necesita información de **estación** y de **calendario**; un único promedio global sería un mal baseline.
2. Si se usa un modelo de conteos, un Poisson simple probablemente se quede corto; conviene contemplar alternativas que admitan sobredispersión (por ejemplo, Binomial Negativa) o modelos de árboles con pérdida para conteos.
3. Las métricas deben elegirse para conteos con muchos ceros (MAE, RMSE); el MAPE no sirve.

### 3.2 Nota de métodos: qué técnicas no se aplican y por qué

El EDA de referencia del curso (notebooks de *Wine Quality*) usa técnicas pensadas para **variables numéricas continuas**. Se evaluaron para este problema y se descartan con justificación:

| Técnica | Por qué no aplica aquí |
|---|---|
| **Regla IQR para atípicos del target** | Con 54 % de ceros, Q1 = 0 y Q3 = 2: el límite superior es 5 y marcaría como "atípicas" al 6 % de las horas, que son simplemente horas pico normales. Los picos se revisan caso por caso (horas máximas). |
| **Mapas de correlación de Pearson/Spearman** | Las variables explicativas disponibles son **categóricas o cíclicas** (estación, hora, día de la semana, mes): una correlación lineal entre las salidas y la hora no tiene sentido (las 23:00 no están "lejos" de las 00:00). Se usan perfiles por grupo (promedio por hora, por día) en su lugar. |
| **Escalamiento o normalización** | No hay modelo todavía; además, cualquier transformación aprendida debe ajustarse solo con datos de entrenamiento después del corte temporal, para evitar fuga de información. |
| **Imputación de faltantes** | La base no tiene faltantes por construcción; los periodos sin operación no se imputan (no se fabrican etiquetas). |
| **Eliminación de picos extremos** | Los picos revisados son demanda real (muchos usuarios distintos, duraciones normales); eliminarlos ocultaría justo la demanda que más importa para el rebalanceo. |

In [ ]:
# Horas con más salidas en todo el histórico
top = base.nlargest(10, "departures")[["station_id", "hour_start", "departures", "day_of_week", "hour"]]
top["estacion"] = top["station_id"].map(stations["snapshot_name"])
top["dia"] = top["hour_start"].dt.day_name()
top[["station_id", "estacion", "hour_start", "dia", "departures"]]

**Interpretación**

Las horas con más salidas del histórico se concentran en la **estación 194 (Av. Vallarta / Av. de Los Arcos)**, en **sábados entre 20:00 y 22:00**. En la hora máxima (2025-09-20 21:00, 207 salidas) se registraron 198 usuarios distintos, con duración mediana de 22 min y destinos variados, y solo 2 viajes con flag: **no parece un error de registro, sino un pico real**, probablemente asociado a un evento o actividad nocturna en la zona (pendiente de confirmar).

**Implicación:** estos picos no deben eliminarse como errores, pero son difíciles de pronosticar sin un calendario de eventos. Se considera en la decisión sobre días atípicos (sección 8) y se contrasta con el análisis de Erik (1.02).

## 4. Patrones temporales (mínimo)

Pendiente. Se completa solo si el notebook 1.01 (José Eduardo) no está disponible a tiempo (plan B).

## 5. Estacionalidad y anomalías (mínimo)

Pendiente. Se completa solo si el notebook 1.02 (Erik) no está disponible a tiempo (plan B).

## 6. Expansión de estaciones (mínimo)

Pendiente. Se completa solo si el notebook 1.03 (José Emmanuel) no está disponible a tiempo (plan B).

## 7. Resultados de los PR del equipo

Pendiente. Resumen y enlace de cada notebook de EDA al integrarse (fecha límite: sábado 24, 15:00 h).

## 8. Decisiones

Pendiente. Para cada punto: **decisión · evidencia · justificación · limitación**.

1. Granularidad estación × hora
2. `flag_short_same_station`
3. `flag_short_cross_station`
4. Ventana histórica que representa la operación actual
5. Toda la historia o ventana reciente
6. 2020–2021
7. Días atípicos
8. Estaciones nuevas
9. `snapshot_capacity` como feature
10. Columnas de `processed_v1`

## 9. `processed_v1`

Pendiente.

## 10. MLflow

Pendiente.

## 11. Conclusiones

Pendiente.